# HotelDataAgentRuntimePublish

Validate the draft HotelOperationsAgent definition and optionally publish the staging configuration
through the Fabric DataAgent runtime API.

In [ ]:
workspace_id = "00000000-0000-0000-0000-000000000000"
data_agent_id = "00000000-0000-0000-0000-000000000000"
capacity_id = "00000000-0000-0000-0000-000000000000"
publish_mode = "validate_only"
published_description = "HotelOperationsAgent draft publication"

In [ ]:
from __future__ import annotations

import importlib
import json
import subprocess
import sys
import time
from uuid import UUID

import sempy.fabric as fabric

client = fabric.FabricRestClient()
base = f"/v1/workspaces/{workspace_id}/dataAgents/{data_agent_id}"

def await_operation(response, operation_name):
    if response.status_code in {200, 201}:
        return response
    if response.status_code != 202:
        raise RuntimeError(f"{operation_name} failed: {response.status_code} {response.text[:500]}")
    location = response.headers.get("Location") or response.headers.get("location")
    if not location:
        raise RuntimeError(f"{operation_name} returned 202 without a Location header")
    retry_after = int(response.headers.get("Retry-After", "5"))
    for _ in range(60):
        time.sleep(max(1, retry_after))
        poll = client.get(location)
        if poll.status_code != 200:
            raise RuntimeError(f"{operation_name} poll failed: {poll.status_code} {poll.text[:500]}")
        payload = poll.json()
        status = str(payload.get("status", "")).lower()
        if status in {"succeeded", "completed"}:
            result_response = client.get(location.rstrip("/") + "/result")
            return result_response if result_response.status_code == 200 else poll
        if status in {"failed", "cancelled"}:
            raise RuntimeError(f"{operation_name} failed: {json.dumps(payload, default=str)}")
        retry_after = int(poll.headers.get("Retry-After", retry_after))
    raise TimeoutError(f"{operation_name} did not complete within the polling limit")

def ensure_data_agent_sdk():
    try:
        from fabric.dataagent.client import FabricDataAgentManagement
        return FabricDataAgentManagement
    except ImportError:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "typing_extensions>=4.13",
                "fabric-data-agent-sdk>=0.1.26a0,<0.2",
            ],
            check=True,
        )
        importlib.invalidate_caches()
        if "typing_extensions" in sys.modules:
            importlib.reload(sys.modules["typing_extensions"])
        from fabric.dataagent.client import FabricDataAgentManagement
        return FabricDataAgentManagement

def workspace_item_id(display_name, item_type):
    response = client.get(f"/v1/workspaces/{workspace_id}/items")
    if response.status_code != 200:
        raise RuntimeError(f"Workspace item discovery failed: {response.status_code} {response.text[:500]}")
    matches = [
        item
        for item in response.json().get("value", [])
        if item.get("displayName") == display_name and item.get("type") == item_type
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {item_type} named {display_name}, found {len(matches)}")
    return matches[0]["id"]

agent_instructions = """You answer hospitality operations and guest-service questions using governed Fabric data only.
HotelOperations is authoritative for properties, room types, rooms, amenities, outlets, inventory, rates, maintenance requests, work orders, and property daily metrics.
HotelGuestServices is authoritative for guests, reservations, stays, room assignments, folios, transactions, payments, adjustments, billing disputes, dining reservations, special requests, action confirmations, and action sagas.
HotelOperationsModel is authoritative for governed OCC, ADR, RevPAR, TRevPAR, ALOS, cancellation, no-show, dispute, request, and maintenance-response KPIs.
HotelOperationsOntology is for relationship traversal only when its entities are available.
For cross-source questions, query each authoritative lakehouse separately and synthesize the result. Use property_id as the shared key between reservations or stays in HotelGuestServices and properties in HotelOperations. Never say the property name or city is unavailable merely because it is in the other lakehouse.
Always resolve every property_id to the official property name from HotelOperations. For chain-wide aggregations, return an overall total and all 20 properties, including zero-count properties, rather than only nonzero groups.
For forward-looking reservation windows, use check_in with status future or arriving and an inclusive date range, and state that interpretation. The source has no booking-created timestamp, so do not label check-in counts as newly created bookings.
Never claim a cross-lakehouse write is ACID. Surface pending_reconciliation from action_sagas when relevant.
Interpret PMS, CRS, RMS, POS, F&B, MICE, OTA, GDS, BAR, FIT, OCC, ADR, RevPAR, TRevPAR, ALOS, OOO, ETA, ETD, VIP, and folio as hotel-industry terms."""

source_configs = {
    "HotelGuestServices": {
        "description": "Authoritative guest-services lakehouse containing guest profiles and all reservation, stay, dining, folio, payment, dispute, request, confirmation, and reconciliation records. Join to HotelOperations by stable business keys such as property_id.",
        "instructions": "Use this source for guest-facing records. reservation_id is the reservation confirmation key. Filter reservations.reservation_id by the full supplied value for exact lookups. Use property_id to obtain property name and city from HotelOperations in a second query. For forward-looking reservation windows, filter check_in inclusively and use status IN ('future','arriving'). The source does not contain a booking-created timestamp.",
        "tables": ["guests", "reservations", "reservation_guests", "stays", "room_assignments", "folios", "folio_transactions", "payments", "adjustments", "billing_disputes", "special_requests", "restaurant_reservations", "action_sagas", "action_confirmations", "idempotency_records"],
        "fewshots": {
            "What is the status and check-in date of reservation RES-DEMO-FAMILY?": "SELECT reservation_id, property_id, check_in, check_out, status FROM reservations WHERE reservation_id = 'RES-DEMO-FAMILY';",
            "Show future reservations arriving at HTL-004.": "SELECT reservation_id, guest_id, property_id, check_in, check_out, status, channel FROM reservations WHERE property_id = 'HTL-004' AND status = 'future' ORDER BY check_in;",
            "Which open billing disputes have the highest amounts?": "SELECT billing_dispute_id, folio_id, status, amount, currency FROM billing_disputes WHERE status NOT IN ('resolved','rejected') ORDER BY amount DESC LIMIT 20;",
            "How many active reservations arrive at each property from 2026-07-12 through 2026-07-18?": "SELECT property_id, COUNT(*) AS reservation_count FROM reservations WHERE check_in BETWEEN DATE '2026-07-12' AND DATE '2026-07-18' AND status IN ('future', 'arriving') GROUP BY property_id ORDER BY reservation_count DESC, property_id;",
        },
    },
    "HotelOperations": {
        "description": "Authoritative hotel-operations lakehouse containing the 20-property roster, room and amenity inventory, outlets, daily inventory and rates, operational KPIs, maintenance requests, and work orders.",
        "instructions": "Use this source for property and operational facts. properties.property_id is the shared key used by HotelGuestServices. For cross-source questions, resolve property_id from the guest record and then query properties for name, city, region, country_code, and currency. For chain-wide aggregations, query the complete property roster so zero-count properties remain in the final answer.",
        "tables": ["properties", "room_types", "rooms", "amenities", "property_amenities", "outlets", "daily_inventory", "daily_rates", "rate_plans", "property_daily_metrics", "maintenance_requests", "work_orders"],
        "fewshots": {
            "How many properties are in the chain?": "SELECT COUNT(*) AS property_count FROM properties;",
            "What are the name and city of property HTL-004?": "SELECT property_id, name, city, region, country_code, currency FROM properties WHERE property_id = 'HTL-004';",
            "Which maintenance requests are still open for HTL-004?": "SELECT maintenance_request_id, property_id, room_id, request_type, priority, status, reported_at FROM maintenance_requests WHERE property_id = 'HTL-004' AND status <> 'resolved' ORDER BY reported_at;",
            "List the complete property roster for a chain-wide result.": "SELECT property_id, name, city, region FROM properties ORDER BY property_id;",
        },
    },
}

definition = client.post(base + "/getDefinition", json={})
definition = await_operation(definition, "DataAgent getDefinition")
result = {"workspace_id": workspace_id, "data_agent_id": data_agent_id, "capacity_id": capacity_id, "publish_mode": publish_mode, "published_description": published_description, "validation": {"definition_status_code": definition.status_code}}
if publish_mode == "deploy":
    FabricDataAgentManagement = ensure_data_agent_sdk()
    management = FabricDataAgentManagement(UUID(data_agent_id), UUID(workspace_id))
    management.update_settings(ai_instructions=agent_instructions)
    synchronized = {}
    for display_name, config in source_configs.items():
        source_id = workspace_item_id(display_name, "Lakehouse")
        try:
            management.delete_staging_datasource(source_id)
        except ValueError:
            pass
        add_response = management.add_staging_datasource(source_id, UUID(workspace_id), type="lakehouse")
        await_operation(add_response, f"Add {display_name} DataAgent source")
        datasource = next(
            source
            for source in management.list_datasources("staging")
            if source.get_configuration("staging").get("id") == source_id
        )
        datasource.update_configuration(
            instructions=config["instructions"],
            description=config["description"],
        )
        roots = datasource.get_elements("staging")["value"]
        schemas = datasource.get_elements("staging", roots[0]["id"])["value"]
        groups = datasource.get_elements("staging", schemas[0]["id"])["value"]
        tables = datasource.get_elements("staging", groups[0]["id"])["value"]
        tables_by_name = {table["displayName"]: table for table in tables}
        missing = sorted(set(config["tables"]) - set(tables_by_name))
        if missing:
            raise RuntimeError(f"{display_name} is missing required tables: {missing}")
        for table_name in config["tables"]:
            datasource.update_element(
                tables_by_name[table_name]["id"],
                is_selected=True,
                description=f"Authorized table: {table_name}.",
            )
        for question, query in config["fewshots"].items():
            datasource.add_fewshot(question, query)
        synchronized[display_name] = {
            "source_id": source_id,
            "selected_table_count": len(config["tables"]),
            "fewshot_count": len(config["fewshots"]),
        }
    publish = management.publish_staging(description=published_description)
    result["synchronized_sources"] = synchronized
    result["publish"] = publish

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))